# Exercise 11 : An Async Subagent on a Server

- Co-deployed graphs share one Agent Server ; the parent starts child work through async task tools
- Lifecycle tools start, check, list and collect ; the parent copies full task ids from state
- Local `langgraph dev` needs enough `--n-jobs-per-worker` for parent and child runs

You will serve an Aurelia supervisor and a diagnosis worker, start a delayed
diagnosis as an async subagent, check it, list it, collect the result, and stop
the server. Laptop only : Colab cannot start the Agent Server. All data is made up.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## Step 1 : Graphs and `langgraph.json`

In [2]:
from pathlib import Path
Path('course_runtime.py').write_text('# Course setup : run this cell first.\n# On the classroom laptop the key comes from .env in the repository root.\n# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.\nimport os\nimport sys\nimport subprocess\nimport warnings\nfrom pathlib import Path\n\nIN_COLAB = "google.colab" in sys.modules\nCOURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"\nROOT = None\n\nif IN_COLAB:\n    # Pinned to match uv.lock ; generated by build/stamp_setup.py\n    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]\n    if not os.getenv("COURSE_SKIP_INSTALL"):\n        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)\n    try:\n        from google.colab import userdata\n        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))\n    except Exception:\n        pass\nelse:\n    from dotenv import load_dotenv\n    for folder in [Path.cwd(), *Path.cwd().parents]:\n        if (folder / "pyproject.toml").exists():\n            ROOT = folder\n            load_dotenv(folder / ".env")\n            break\n\nPROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"\nif PROVIDER == "ollama":\n    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")\n    BASE_URL = "http://localhost:11434/v1"\n    API_KEY = "ollama"\n    THINKING_OFF = {"reasoning_effort": "none"}\nelse:\n    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up\n    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")\n    API_KEY = os.getenv("COURSE_API_KEY")\n    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;\n    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider\n    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}\n    if not API_KEY:\n        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")\n\nfrom langchain_core._api import LangChainBetaWarning\nwarnings.filterwarnings("ignore", category=LangChainBetaWarning)\n\nfrom langchain_openai import ChatOpenAI\nfrom langchain_core.rate_limiters import InMemoryRateLimiter\n\n# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.\n# Paid OpenRouter models have no platform request cap (measured 26 October 2026)\nRATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),\n                                   check_every_n_seconds=0.1, max_bucket_size=1)\n\n\ndef make_llm(**kw):\n    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""\n    kw.setdefault("model", MODEL)\n    if PROVIDER == "ollama":\n        kw.setdefault("temperature", 0)\n    else:\n        kw.setdefault("seed", 7)\n    kw.setdefault("extra_body", THINKING_OFF)\n    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging\n    kw.setdefault("rate_limiter", RATE_LIMITER)\n    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)\n\n\ndef make_embeddings(**kw):\n    """Local sentence embeddings, so retrieval costs no API calls."""\n    from langchain_huggingface import HuggingFaceEmbeddings\n    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")\n    return HuggingFaceEmbeddings(**kw)\n\n\ndef data(name):\n    """Path to a course data file ; downloaded when there is no local copy."""\n    if ROOT and (ROOT / "data" / name).exists():\n        return str(ROOT / "data" / name)\n    target = Path(name)\n    if not target.exists():\n        import urllib.request\n        target.parent.mkdir(parents=True, exist_ok=True)\n        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)\n    return str(target)\n\n\ndef data_folder(name):\n    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""\n    if ROOT and (ROOT / "data" / name).is_dir():\n        return str(ROOT / "data" / name)\n    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():\n        data(f"{name}/{file}")\n    return name\n\n\nllm = make_llm()\nprint(f"model {MODEL} via {PROVIDER}")\n', encoding='utf-8')
Path('graphs.py').write_text('"""Aurelia diagnosis graphs for exercise 11."""\nimport asyncio\nimport json\nimport sqlite3\nfrom pathlib import Path\n\nfrom course_runtime import data, data_folder, llm\nfrom deepagents import AsyncSubAgent, create_deep_agent\nfrom langchain.agents import create_agent\nfrom langchain.tools import tool\nfrom langgraph.graph import END, START, MessagesState, StateGraph\nfrom typing_extensions import NotRequired\n\nDB = Path(data("capstone/service.db")).resolve()\nMANUALS = Path(data_folder("capstone/manuals"))\n\n\n@tool\ndef part_stock(part_no: str) -> str:\n    """Look up an Aurelia part\'s stock and depot by its part number."""\n    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:\n        rows = db.execute("SELECT part_no, stock, depot FROM parts WHERE part_no = ?",\n                          (part_no.upper(),)).fetchall()\n    return json.dumps(rows)\n\n\n@tool\ndef fault_guidance(error_code: str) -> str:\n    """Read the meaning and first action for an Aurelia error code."""\n    return "\\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))\n                     for line in p.read_text(encoding="utf-8").splitlines()\n                     if line.startswith("| " + error_code.upper() + " |")) or "No matching code."\n\n\nSYSTEM = ("Support Aurelia field engineers. Use tools for facts. Preserve stock, "\n          "depot and first actions. Be brief.")\nworker = create_agent(llm, tools=[part_stock, fault_guidance], system_prompt=SYSTEM, name="worker")\n\n\nclass WorkState(MessagesState):\n    delay_seconds: NotRequired[float]\n\n\nasync def diagnose(state):\n    text = str(state["messages"][-1].content)\n    # Sleep unless the task says NO DELAY. Do not require the word DELAY : the\n    # supervisor may paraphrase the launch text and drop it.\n    if "NO DELAY" not in text:\n        await asyncio.sleep(120)\n    result = await worker.ainvoke({"messages": state["messages"]})\n    return {"messages": result["messages"]}\n\n\nbuilder = StateGraph(WorkState)\nbuilder.add_node("diagnose", diagnose)\nbuilder.add_edge(START, "diagnose")\nbuilder.add_edge("diagnose", END)\ndiagnosis = builder.compile()\n\nsupervisor = create_deep_agent(\n    model=llm,\n    subagents=[AsyncSubAgent(name="diagnosis", graph_id="diagnosis",\n                             description="Long Aurelia fault diagnosis with manuals and parts.")],\n    system_prompt=(\n        "Coordinate Aurelia diagnosis. Use only the requested async task tool, then "\n        "respond briefly. After launch return to the user without checking. Copy full "\n        "task IDs from async_tasks. Always fetch status before reporting it."\n    ),\n    name="supervisor",\n)\n', encoding='utf-8')
config = {'dependencies': ['.'], 'graphs': {'diagnosis': './graphs.py:diagnosis', 'supervisor': './graphs.py:supervisor'}, 'env': '../.env'}
Path('langgraph.json').write_text(__import__('json').dumps(config, indent=2), encoding='utf-8')
print('wrote course_runtime.py, graphs.py, langgraph.json')
print(__import__('json').dumps(config, indent=2))

wrote course_runtime.py, graphs.py, langgraph.json
{
  "dependencies": [
    "."
  ],
  "graphs": {
    "diagnosis": "./graphs.py:diagnosis",
    "supervisor": "./graphs.py:supervisor"
  },
  "env": "../.env"
}


In [3]:
print("check :", "PASS" if Path("graphs.py").exists() and Path("langgraph.json").exists() else "FAIL")
assert Path("graphs.py").exists() and Path("langgraph.json").exists()

check : PASS


## Step 2 : Start the server

In [4]:
import atexit
import os
import signal
import socket
import subprocess
import time
from pathlib import Path

import httpx
from langgraph_sdk import get_sync_client

HERE = Path.cwd()
servers = []


def port_open(port):
    with socket.socket() as probe:
        probe.settimeout(0.5)
        return probe.connect_ex(("127.0.0.1", port)) == 0


def wait_port_free(port, seconds=60):
    for _ in range(seconds * 2):
        if not port_open(port):
            return
        time.sleep(0.5)
    raise TimeoutError(f"Port {port} stayed busy after stop.")


def stop_server(proc):
    if proc is None:
        return
    port = getattr(proc, "course_port", None)
    was_running = proc.poll() is None
    if was_running:
        proc.send_signal(signal.CTRL_BREAK_EVENT if os.name == "nt" else signal.SIGINT)
        try:
            proc.wait(timeout=25)
        except subprocess.TimeoutExpired:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(proc.pid), "/T", "/F"],
                               capture_output=True, check=False)
            else:
                proc.kill()
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                pass
    if was_running and port is not None:
        wait_port_free(port, seconds=90)


def start_server(port):
    if port not in (2134, 2135):
        raise ValueError("Use an exercises 08-11 port (2134 or 2135).")
    wait_port_free(port, seconds=90)
    log = (HERE / f"server-{port}.log").open("w", encoding="utf-8")
    env = dict(os.environ, PYTHONIOENCODING="utf-8", LANGGRAPH_CLI_NO_ANALYTICS="1")
    env["LANGSMITH_TRACING"] = "false"
    proc = subprocess.Popen(
        ["uv", "run", "langgraph", "dev", "--no-browser", "--no-reload", "--port", str(port),
         "--n-jobs-per-worker", "8", "--studio-url", "https://eu.smith.langchain.com"],
        cwd=HERE, env=env, stdout=log, stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0)
    proc.course_port = port
    log.close()
    servers.append(proc)
    atexit.register(stop_server, proc)
    for _ in range(180):
        if proc.poll() is not None:
            raise RuntimeError((HERE / f"server-{port}.log").read_text(encoding="utf-8")[-5000:])
        try:
            response = httpx.get(f"http://127.0.0.1:{port}/ok", timeout=1)
            if response.is_success:
                return proc
        except httpx.TransportError:
            pass
        time.sleep(0.5)
    stop_server(proc)
    raise TimeoutError("Agent Server did not become ready.")


def connect(port):
    return get_sync_client(url=f"http://127.0.0.1:{port}", api_key=None, timeout=240)


def clear_dev_storage():
    import shutil
    target = HERE / ".langgraph_api"
    if target.exists():
        shutil.rmtree(target)
        print("cleared :", target.name)

In [5]:
clear_dev_storage()
server = start_server(2134)
client = connect(2134)
print("ready :", httpx.get("http://127.0.0.1:2134/ok").json())
print("assistants :", [(a["name"], a["graph_id"]) for a in client.assistants.search()])

cleared : .langgraph_api


ready : {'ok': True, 'services': {}}
assistants : [('supervisor', 'supervisor'), ('diagnosis', 'diagnosis')]


In [6]:
names = {a["graph_id"] for a in client.assistants.search()}
print("check :", "PASS" if {"diagnosis", "supervisor"} <= names else "FAIL", names)
assert {"diagnosis", "supervisor"} <= names

check : PASS {'supervisor', 'diagnosis'}


## Step 3 : Start, check and list

Start a delayed diagnosis (120 s hold), then check and list it. The first check
should run while the task is still `running` ; print that status.

In [7]:
tid = client.threads.create()["thread_id"]
used = set()


def supervise(text):
    result = client.runs.wait(tid, "supervisor", input={"messages": [{"role": "user", "content": text}]},
                              config={"recursion_limit": 30}, raise_error=True)
    if "messages" not in result:
        raise RuntimeError("supervisor run finished without messages : " + repr(result)[:800])
    print("response :", result["messages"][-1]["content"])
    print("tasks :", result.get("async_tasks", {}))
    used.update(c["name"] for m in result["messages"] for c in m.get("tool_calls", []) or [])
    return result


out = supervise(
    "Use start_async_task for diagnosis. Task: DELAY then diagnose E-CT-110 and look up "
    "CT-FN-02 stock. Return the full task ID and stop. Do not check yet."
)
task_id = next(iter(out["async_tasks"]))
print("task ID :", task_id)
out = supervise(f"Use check_async_task for {task_id}. Report its fresh status and stop.")
first_status = out["async_tasks"][task_id]["status"]
print("first check status :", first_status)
out = supervise("Use list_async_tasks with status_filter all. Report the current list and stop.")
print("lifecycle so far :", sorted(used))
assert {"start_async_task", "check_async_task", "list_async_tasks"} <= used
assert first_status == "running"

response : Task started: `01a17a53-76b8-7be3-b80d-56a8dbbeffe6`
tasks : {'01a17a53-76b8-7be3-b80d-56a8dbbeffe6': {'task_id': '01a17a53-76b8-7be3-b80d-56a8dbbeffe6', 'agent_name': 'diagnosis', 'thread_id': '01a17a53-76b8-7be3-b80d-56a8dbbeffe6', 'run_id': '01a17a53-76bc-7293-a46e-220629378eee', 'status': 'running', 'created_at': '2026-10-26T22:27:32Z', 'last_checked_at': '2026-10-26T22:27:35Z', 'last_updated_at': '2026-10-26T22:27:35Z'}}
task ID : 01a17a53-76b8-7be3-b80d-56a8dbbeffe6


response : Fresh status: running.
tasks : {'01a17a53-76b8-7be3-b80d-56a8dbbeffe6': {'task_id': '01a17a53-76b8-7be3-b80d-56a8dbbeffe6', 'agent_name': 'diagnosis', 'thread_id': '01a17a53-76b8-7be3-b80d-56a8dbbeffe6', 'run_id': '01a17a53-76bc-7293-a46e-220629378eee', 'status': 'running', 'created_at': '2026-10-26T22:27:32Z', 'last_checked_at': '2026-10-26T22:27:37Z', 'last_updated_at': '2026-10-26T22:27:35Z'}}
first check status : running


response : Current async tasks:
- `01a17a53-76b8-7be3-b80d-56a8dbbeffe6` — diagnosis — running
tasks : {'01a17a53-76b8-7be3-b80d-56a8dbbeffe6': {'task_id': '01a17a53-76b8-7be3-b80d-56a8dbbeffe6', 'agent_name': 'diagnosis', 'thread_id': '01a17a53-76b8-7be3-b80d-56a8dbbeffe6', 'run_id': '01a17a53-76bc-7293-a46e-220629378eee', 'status': 'running', 'created_at': '2026-10-26T22:27:32Z', 'last_checked_at': '2026-10-26T22:27:42Z', 'last_updated_at': '2026-10-26T22:27:35Z'}}
lifecycle so far : ['check_async_task', 'list_async_tasks', 'start_async_task']


In [8]:
print("first check status :", first_status)
print("check :", "PASS" if first_status == "running" and {"start_async_task", "check_async_task", "list_async_tasks"} <= used else "FAIL")
assert first_status == "running"
assert {"start_async_task", "check_async_task", "list_async_tasks"} <= used

first check status : running
check : PASS


## Step 4 : Collect the result

Join the child run, check again, confirm the diagnosis facts, and stop the server.

In [9]:
task = out["async_tasks"][task_id]
client.runs.join(task_id, task["run_id"])
out = supervise(f"Use check_async_task for {task_id}. Return the result, first action and stock.")
status = out["async_tasks"][task_id]["status"]
print("final status :", status)
text = str(out["messages"][-1]["content"]).lower()
print("answer text :", text[:500])
assert status == "success"
assert "6" in text and "houten" in text and "cooling fan" in text

response : **Result:** DELTA identifies E-CT-110 as gantry over-temperature.  
**First action:** Check the gantry cooling fan; replace it if airflow is below 80%.  
**CT-FN-02 stock:** 6 units at the Houten depot.
tasks : {'01a17a53-76b8-7be3-b80d-56a8dbbeffe6': {'task_id': '01a17a53-76b8-7be3-b80d-56a8dbbeffe6', 'agent_name': 'diagnosis', 'thread_id': '01a17a53-76b8-7be3-b80d-56a8dbbeffe6', 'run_id': '01a17a53-76bc-7293-a46e-220629378eee', 'status': 'success', 'created_at': '2026-10-26T22:27:32Z', 'last_checked_at': '2026-10-26T22:29:39Z', 'last_updated_at': '2026-10-26T22:29:37Z'}}
final status : success
answer text : **result:** delta identifies e-ct-110 as gantry over-temperature.  
**first action:** check the gantry cooling fan; replace it if airflow is below 80%.  
**ct-fn-02 stock:** 6 units at the houten depot.


In [10]:
for proc in servers:
    stop_server(proc)
print("servers stopped :", all(proc.poll() is not None for proc in servers))
print("check :", "PASS" if status == "success" else "FAIL")
assert status == "success"

servers stopped : True
check : PASS


## If time permits

- Cancel a second delayed task before it finishes
- Steer the running task with `update_async_task` to look up UX-BT-02 instead
- Restart the server in the same folder and read whether the thread still lists the task